# Deep Momentum — Run from Cached Data

Skips the FMP fetch step. Reads parquet files from `data/` directly.
Use this on RunPod or any machine where the data is already available.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
os.chdir(os.path.dirname(os.path.abspath('__file__')))

PILOT_COUNTRIES = ['TO', 'OL', 'AT']  # Canada, Norway, Austria
N_ENSEMBLE = 100  # paper uses 100
OOS_START = '2010-01-31'

print('Countries:', PILOT_COUNTRIES)
print(f'Ensemble: {N_ENSEMBLE}')
print(f'OOS start: {OOS_START}')

## Check data availability

In [ ]:
from pathlib import Path
from config import DATA_DIR, CACHE_DIR, RESULTS_DIR, COUNTRIES
import pandas as pd

data_dir = Path(DATA_DIR)
cache_dir = Path(CACHE_DIR)

for suffix in PILOT_COUNTRIES:
    f = data_dir / f'monthly_{suffix}.parquet'
    if f.exists():
        df = pd.read_parquet(f)
        _, name, _, _ = COUNTRIES[suffix]
        print(f'{name:20s} {df["symbol"].nunique():>5d} stocks  '
              f'{len(df):>7d} obs  '
              f'{df["date"].min().date()} to {df["date"].max().date()}')
    else:
        print(f'MISSING: {f}')

## Step 2: Filter

In [ ]:
from data_filter import filter_country, check_country_eligibility

for suffix in PILOT_COUNTRIES:
    _, country_name, _, _ = COUNTRIES[suffix]
    f = data_dir / f'monthly_{suffix}.parquet'
    if not f.exists():
        print(f'No data for {suffix}')
        continue
    df = pd.read_parquet(f)
    filtered = filter_country(df, country_name)
    filtered['country_suffix'] = suffix
    filtered.to_parquet(cache_dir / f'filtered_{suffix}.parquet', index=False)
    eligible, reason = check_country_eligibility(filtered, suffix)
    print(f'  Eligible: {eligible} — {reason}')
    print(f'  Saved: {cache_dir / f"filtered_{suffix}.parquet"}')

## Step 3: Features

In [ ]:
from features import build_features

features_data = {}

for suffix in PILOT_COUNTRIES:
    filtered_path = cache_dir / f'filtered_{suffix}.parquet'
    if not filtered_path.exists():
        print(f'No filtered data for {suffix}, skipping')
        continue
    
    _, country_name, _, _ = COUNTRIES[suffix]
    df = pd.read_parquet(filtered_path)
    df, feature_cols = build_features(df, country_name)
    
    df.to_parquet(cache_dir / f'features_{suffix}.parquet', index=False)
    features_data[suffix] = (df, feature_cols)

print(f'\nFeatures built for {len(features_data)} countries')
print(f'Feature columns ({len(feature_cols)}): {feature_cols}')

## Step 4: Model Training + Prediction

This is the slow step — 100 XGBoost fits per retraining, ~20-30 retrainings per country.

In [ ]:
from model import run_walk_forward

predictions_data = {}

for suffix in PILOT_COUNTRIES:
    if suffix not in features_data:
        continue
    
    _, country_name, _, _ = COUNTRIES[suffix]
    df, feature_cols = features_data[suffix]
    
    print(f'\n{"="*60}')
    print(f'TRAINING: {country_name} (.{suffix})')
    print(f'{"="*60}')
    
    predictions = run_walk_forward(
        df, feature_cols,
        n_ensemble=N_ENSEMBLE,
        verbose=True,
    )
    
    if not predictions.empty:
        predictions.to_parquet(cache_dir / f'predictions_{suffix}.parquet', index=False)
        predictions_data[suffix] = predictions
        print(f'  Saved {len(predictions)} predictions')
    else:
        print(f'  No predictions generated')

print(f'\nPredictions generated for {len(predictions_data)} countries')

## Step 5: Portfolio Construction

In [ ]:
from portfolio import run_all_strategies, print_performance_table

all_results = {}

for suffix in PILOT_COUNTRIES:
    if suffix not in features_data or suffix not in predictions_data:
        continue
    
    _, country_name, _, _ = COUNTRIES[suffix]
    df, feature_cols = features_data[suffix]
    predictions = predictions_data[suffix]
    
    # Merge fwd_return into predictions
    fwd = df[['symbol', 'date', 'fwd_return']].dropna()
    predictions = predictions.drop(columns=['fwd_return'], errors='ignore')
    predictions = predictions.merge(fwd, on=['symbol', 'date'], how='left')
    
    print(f'\n{"="*60}')
    print(f'{country_name}')
    print(f'{"="*60}')
    
    results = run_all_strategies(df, predictions, oos_start=OOS_START)
    print_performance_table(results)
    all_results[suffix] = results

## Step 6: Full Report

In [ ]:
from metrics import full_report

for suffix in PILOT_COUNTRIES:
    if suffix not in features_data or suffix not in predictions_data:
        continue
    
    _, country_name, _, _ = COUNTRIES[suffix]
    df, feature_cols = features_data[suffix]
    predictions = predictions_data[suffix]
    
    fwd = df[['symbol', 'date', 'fwd_return']].dropna()
    preds = predictions.drop(columns=['fwd_return'], errors='ignore')
    preds = preds.merge(fwd, on=['symbol', 'date'], how='left')
    
    full_report(df, preds, all_results.get(suffix, {}), country_name)

## Plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

for suffix in PILOT_COUNTRIES:
    if suffix not in all_results:
        continue
    
    _, country_name, _, _ = COUNTRIES[suffix]
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    ax = axes[0]
    for name, color in [('MOM', '#1f77b4'), ('XGB', '#ff7f0e'), ('RET', '#2ca02c'), ('SRP', '#d62728')]:
        if name in all_results[suffix] and not all_results[suffix][name]['portfolio'].empty:
            port = all_results[suffix][name]['portfolio']
            cum = (1 + port['ls_ret']).cumprod()
            ax.plot(port['date'], cum, label=name, color=color, linewidth=1.2)
    
    ax.set_ylabel('Cumulative Return (1 = start)')
    ax.set_title(f'{country_name} — Cumulative L/S Returns')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_yscale('log')
    
    ax = axes[1]
    for name, color in [('MOM', '#1f77b4'), ('XGB', '#ff7f0e'), ('RET', '#2ca02c')]:
        if name in all_results[suffix] and not all_results[suffix][name]['portfolio'].empty:
            port = all_results[suffix][name]['portfolio']
            rolling = port['ls_ret'].rolling(12).mean() * 12
            ax.plot(port['date'], rolling, label=f'{name} (rolling 12m ann.)',
                    color=color, linewidth=1)
    
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_ylabel('Annualized Return')
    ax.set_title(f'{country_name} — Rolling 12-Month Return')
    ax.legend()
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'results/{suffix}_performance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print()

## Cross-Country Summary

In [ ]:
print(f'{"Country":<15s}  {"MOM Ret":>8s} {"MOM SR":>7s}  {"XGB Ret":>8s} {"XGB SR":>7s}  {"RET Ret":>8s} {"RET SR":>7s}')
print('-' * 70)

for suffix in PILOT_COUNTRIES:
    if suffix not in all_results:
        continue
    _, country_name, _, _ = COUNTRIES[suffix]
    
    parts = [f'{country_name:<15s}']
    for name in ['MOM', 'XGB', 'RET']:
        if name in all_results[suffix] and all_results[suffix][name]['metrics']:
            m = all_results[suffix][name]['metrics']
            parts.append(f' {m["mean_annual"]:>7.1%} {m["sharpe"]:>7.3f}')
        else:
            parts.append(f' {"N/A":>7s} {"N/A":>7s}')
    
    print(''.join(parts))